In [ ]:
import os
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
class TaskVector:
    def __init__(self, pretrained_model, finetuned_model):
        # task vector
        self.task_vector_param_dict = {}
        pretrained_param_dict = {param_name: param_value
                for param_name, param_value in pretrained_model.named_parameters()}
        finetuned_param_dict = {param_name: param_value
                for param_name, param_value in finetuned_model.named_parameters()}

        # 这里可以排除一些参数不做合并
        param_names_to_merge = list(pretrained_param_dict.keys())
        with torch.no_grad():
            for param_name in param_names_to_merge:
                self.task_vector_param_dict[param_name] = finetuned_param_dict[param_name] - pretrained_param_dict[param_name]

    def combine_with_pretrained_model(self, pretrained_model, scaling_coefficient=0.5):
        """模型参数的合并，求和
        """
        pretrained_param_dict = {param_name: param_value
                for param_name, param_value in pretrained_model.named_parameters()} 
        with torch.no_grad():
            merged_params = {}
            for param_name in self.task_vector_param_dict.keys():
                merged_params[param_name] = pretrained_param_dict[param_name] + scaling_coefficient * self.task_vector_param_dict[param_name]

        return merged_params

In [ ]:
def merging_models(merged_model, models_to_merge, scaling_coefficient):
    """合并模型参数
       merged_model: Base Model类
       models_to_merge： list[Model类]
    """

    merged_task_vector = None
    # 可以合并多个模型
    while len(models_to_merge) > 0:
        if merged_task_vector is None:
            merged_task_vector = TaskVector(
                pretrained_model=merged_model,
                finetuned_model=models_to_merge.pop(0)
            )
        else:
            merged_task_vector += TaskVector(
                pretrained_model=merged_model,
                finetuned_model=models_to_merge.pop(0)
            )
    # 执行合并
    merged_params = merged_task_vector.combine_with_pretrained_model(
         pretrained_model=merged_model,
        scaling_coefficient=scaling_coefficient
    )

    # 把合并后的参数字典拷贝回原模型
    for param_name, param_value in merged_model.named_parameters():
        if param_name in merged_params:
            param_value.data.copy_(merged_params[param_name])

    return merged_model